<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/01_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Cell 1 — connect Google Drive, where the data will live
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/nbi'
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)
print(os.listdir(DATA_DIR))

Mounted at /content/drive
['raw', 'nbi.duckdb', 'nbi.duckdb.wal']


In [4]:
# Cell 2 — clone the repo (or update it if already cloned) and install the libraries
import os
REPO_DIR = '/content/bridge-deterioration-nbi'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q https://github.com/AHichem/bridge-deterioration-nbi.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt

/content/bridge-deterioration-nbi
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 890.5/890.5 kB 41.5 MB/s eta 0:00:00


In [41]:
import requests, zipfile
from pathlib import Path
import pandas as pd

In [ ]:
RAW_DIR = Path(DATA_DIR) / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
YEARS = range(2015, 2026)   # 2015..2025

In [38]:
#def url_for(year):
#    return f"https://www.fhwa.dot.gov/bridge/nbi/{year}hwybronefiledel.zip"


def url_for(year):
    base = "https://www.fhwa.dot.gov/bridge/nbi/"
    for name in [f"{year}hwybronefiledel.zip", f"{year}hwybronlyonefile.zip"]:
        r = requests.head(base + name, headers={"User-Agent": "Mozilla/5.0"},
                          allow_redirects=True, timeout=30)
        if r.status_code == 200:
            return base + name
    raise ValueError(f"{year}: no file found")

In [39]:
def download_year(year):
    zip_path = RAW_DIR / f"{year}.zip"
    txt_path = RAW_DIR / f"nbi_{year}.txt"

    # 1. Skip if already done
    if txt_path.exists():
        print(f"{year}: already there, skipping")
        return txt_path

    # 2. Download the zip in chunks
    print(f"{year}: downloading...")
    with requests.get(url_for(year), stream=True, timeout=60,
                      headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)

    # 3. Unzip: look inside first
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        print(f"{year}: zip contains {names}")
        inner_name = [n for n in names if n.endswith(".txt")][0]
        z.extract(inner_name, RAW_DIR)

    # 4. Rename to a predictable name, delete the zip
    (RAW_DIR / inner_name).rename(txt_path)
    zip_path.unlink()

    print(f"{year}: done → {txt_path.name}")
    return txt_path

In [40]:
for y in YEARS:
  ddl = download_year(y)
  print(f"For the {y} size is:  {ddl.stat().st_size / 1024**2} MB")
  with open(ddl, encoding="latin-1") as f: n_lines = sum(1 for _ in f)-1
  print(f"For the {y} nb line is: {n_lines} lignes")

2015: already there, skipping
For the 2015 size is:  0.020605087280273438 MB
For the 2015 nb line is: 125 lignes
2016: downloading...
2016: zip contains ['slubkin_992016-20170106140351.txt']
2016: done → nbi_2016.txt
For the 2016 size is:  259.24455642700195 MB
For the 2016 nb line is: 614387 lignes
2017: downloading...
2017: zip contains ['2017HwyBridgesDelimitedAllStates.txt']
2017: done → nbi_2017.txt
For the 2017 size is:  264.4661617279053 MB
For the 2017 nb line is: 615002 lignes
2018: downloading...
2018: zip contains ['2018HwyBridgesDelimitedAllStates.txt']
2018: done → nbi_2018.txt
For the 2018 size is:  264.90919399261475 MB
For the 2018 nb line is: 616096 lignes
2019: downloading...
2019: zip contains ['2019HwyBridgesDelimitedAllStates.txt']
2019: done → nbi_2019.txt
For the 2019 size is:  241.84528827667236 MB
For the 2019 nb line is: 617084 lignes
2020: already there, skipping
For the 2020 size is:  242.32261753082275 MB
For the 2020 nb line is: 618456 lignes
2021: already

In [44]:
for y in YEARS:
  path = download_year(y)
  KEEP=[]
  KEEP.append(pd.read_csv(path, nrows=0, quotechar="'", encoding="latin-1"))

2015: already there, skipping
2016: already there, skipping
2017: already there, skipping
2018: already there, skipping
2019: already there, skipping
2020: already there, skipping
2021: already there, skipping
2022: already there, skipping
2023: already there, skipping
2024: already there, skipping
2025: already there, skipping


In [46]:
KEEP = [
    "STRUCTURE_NUMBER_008",
    "STATE_CODE_001",
    "YEAR_BUILT_027",
    "DECK_COND_058",
    "SUPERSTRUCTURE_COND_059",
    "SUBSTRUCTURE_COND_060",
]

for y in YEARS:
    path = RAW_DIR / f"nbi_{y}.txt"
    cols = pd.read_csv(path, nrows=0, quotechar="'", encoding="latin-1").columns
    missing = set(KEEP) - set(cols)
    print(y, "manquantes :", sorted(missing) if missing else "aucune")

2015 manquantes : ['DECK_COND_058', 'STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'SUBSTRUCTURE_COND_060', 'SUPERSTRUCTURE_COND_059', 'YEAR_BUILT_027']
2016 manquantes : aucune
2017 manquantes : aucune
2018 manquantes : aucune
2019 manquantes : aucune
2020 manquantes : aucune
2021 manquantes : aucune
2022 manquantes : aucune
2023 manquantes : aucune
2024 manquantes : aucune
2025 manquantes : aucune
